# Notebook 01 - Knowledge Graphs & ETL Pipeline
dical knowledge graph from scratch: schema design, Cypher queries, batch ingestion, and LLM-powered entity extraction.



In [1]:
import sys, os
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv()

from utils import validate, dump, get_neo4j_client, get_general_llm, get_strong_llm, get_embeddings
validate()

## 1. Connect to Neo4j

In [2]:
client = get_neo4j_client()
client.verify_connectivity()

2026-04-05 15:27:43.231 | SUCCESS  | utils.neo4j_client:verify_connectivity:35 - Connected to Neo4j at neo4j+s://c560929d.databases.neo4j.io


True

## 2. Graph Schema Design

Our domain: **Drug Interactions**. We model 4 entity types and 5 relationship types.

In [3]:
NODE_LABELS = {
    "Drug": "A pharmacological agent (e.g., Warfarin, Aspirin)",
    "Condition": "A medical condition or disease (e.g., Diabetes, Hypertension)",
    "SideEffect": "An adverse effect caused by a drug (e.g., Bleeding, Nausea)",
    "DrugClass": "A pharmacological class (e.g., NSAID, Anticoagulant, SSRI)",
}

RELATIONSHIP_TYPES = {
    "INTERACTS_WITH": "Drug-Drug interaction (bidirectional in meaning, stored as directed edge)",
    "TREATS": "Drug treats a Condition",
    "CAUSES_SIDE_EFFECT": "Drug causes a SideEffect",
    "CONTRAINDICATED_FOR": "Drug is contraindicated for a Condition",
    "BELONGS_TO_CLASS": "Drug belongs to a DrugClass",
}

ALLOWED_REL_TYPES = set(RELATIONSHIP_TYPES.keys())

print("Node Labels:")
for label, desc in NODE_LABELS.items():
    print(f"  :{label} — {desc}")
print("\nRelationship Types:")
for rel, desc in RELATIONSHIP_TYPES.items():
    print(f"  [{rel}] — {desc}")

Node Labels:
  :Drug — A pharmacological agent (e.g., Warfarin, Aspirin)
  :Condition — A medical condition or disease (e.g., Diabetes, Hypertension)
  :SideEffect — An adverse effect caused by a drug (e.g., Bleeding, Nausea)
  :DrugClass — A pharmacological class (e.g., NSAID, Anticoagulant, SSRI)

Relationship Types:
  [INTERACTS_WITH] — Drug-Drug interaction (bidirectional in meaning, stored as directed edge)
  [TREATS] — Drug treats a Condition
  [CAUSES_SIDE_EFFECT] — Drug causes a SideEffect
  [CONTRAINDICATED_FOR] — Drug is contraindicated for a Condition
  [BELONGS_TO_CLASS] — Drug belongs to a DrugClass


## 3. Constraints, Indexes & Sample Data

In [4]:
CONSTRAINT_QUERIES = [
    "CREATE CONSTRAINT drug_name IF NOT EXISTS FOR (d:Drug) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT condition_name IF NOT EXISTS FOR (c:Condition) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT side_effect_name IF NOT EXISTS FOR (s:SideEffect) REQUIRE s.name IS UNIQUE",
    "CREATE CONSTRAINT drug_class_name IF NOT EXISTS FOR (dc:DrugClass) REQUIRE dc.name IS UNIQUE",
]

INDEX_QUERIES = [
    "CREATE INDEX drug_name_idx IF NOT EXISTS FOR (d:Drug) ON (d.name)",
    "CREATE INDEX condition_name_idx IF NOT EXISTS FOR (c:Condition) ON (c.name)",
    "CREATE INDEX side_effect_name_idx IF NOT EXISTS FOR (s:SideEffect) ON (s.name)",
    "CREATE INDEX drug_class_name_idx IF NOT EXISTS FOR (dc:DrugClass) ON (dc.name)",
]

# Clean slate for demo
client.write("MATCH (n) DETACH DELETE n")

for q in CONSTRAINT_QUERIES + INDEX_QUERIES:
    try:
        client.write(q)
    except Exception as e:
        print(f"  (may already exist): {e}")

print("Schema setup complete!")

# Create a few drugs, conditions, and relationships
sample_cypher = """
CREATE (warfarin:Drug {name: 'Warfarin', description: 'Anticoagulant medication'})
CREATE (aspirin:Drug {name: 'Aspirin', description: 'NSAID and antiplatelet agent'})
CREATE (ibuprofen:Drug {name: 'Ibuprofen', description: 'Non-steroidal anti-inflammatory drug'})
CREATE (bleeding:SideEffect {name: 'Increased Bleeding Risk'})
CREATE (dvt:Condition {name: 'Deep Vein Thrombosis'})
CREATE (anticoag:DrugClass {name: 'Anticoagulant'})
CREATE (nsaid:DrugClass {name: 'NSAID'})

CREATE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Additive anticoagulant effect'}]->(aspirin)
CREATE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Increased bleeding via platelet and coagulation pathway'}]->(ibuprofen)
CREATE (warfarin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
CREATE (aspirin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
CREATE (warfarin)-[:TREATS]->(dvt)
CREATE (warfarin)-[:BELONGS_TO_CLASS]->(anticoag)
CREATE (aspirin)-[:BELONGS_TO_CLASS]->(nsaid)
CREATE (ibuprofen)-[:BELONGS_TO_CLASS]->(nsaid)
"""
client.write(sample_cypher)
print("Sample data created!")

Schema setup complete!
Sample data created!


> **Try in Neo4j Browser** — paste these queries to see the sample data:
>
> ```cypher
> -- See all nodes and relationships
> MATCH (n)-[r]->(m) RETURN n, r, m
> ```
>
> ```cypher
> -- Check constraints
> SHOW CONSTRAINTS
> ```
>
> ```cypher
> -- Check indexes
> SHOW INDEXES
> ```

## 4. Cypher Queries — Hands-On

In [5]:
# Find all drugs
results = client.query("MATCH (d:Drug) RETURN d.name AS name, d.description AS desc")
print("All Drugs:")
for r in results:
    print(f"  {r['name']}: {r['desc']}")

# Find what Warfarin interacts with
print("\nWarfarin interactions:")
results = client.query("""
    MATCH (d:Drug {name: 'Warfarin'})-[r:INTERACTS_WITH]-(other:Drug)
    RETURN other.name AS drug, r.severity AS severity, r.mechanism AS mechanism
""")
for r in results:
    print(f"  {r['drug']} — Severity: {r['severity']}, Mechanism: {r['mechanism']}")

All Drugs:
  Warfarin: Anticoagulant medication
  Aspirin: NSAID and antiplatelet agent
  Ibuprofen: Non-steroidal anti-inflammatory drug

Warfarin interactions:
  Aspirin — Severity: Major, Mechanism: Additive anticoagulant effect
  Ibuprofen — Severity: Major, Mechanism: Increased bleeding via platelet and coagulation pathway


> **Try in Neo4j Browser:**
>
> ```cypher
> -- All drugs
> MATCH (d:Drug) RETURN d.name AS name, d.description AS desc
> ```
>
> ```cypher
> -- Warfarin interactions with severity and mechanism
> MATCH (d:Drug {name: 'Warfarin'})-[r:INTERACTS_WITH]-(other:Drug)
> RETURN other.name AS drug, r.severity AS severity, r.mechanism AS mechanism
> ```

In [6]:
drug_name = "Warfarin"
results = client.query(
    "MATCH (d:Drug {name: $name})-[r]->(m) RETURN type(r) AS rel, m.name AS target",
    {"name": drug_name}
)
print(f"All relationships FROM {drug_name}:")
for r in results:
    print(f"  --[{r['rel']}]--> {r['target']}")

All relationships FROM Warfarin:
  --[INTERACTS_WITH]--> Aspirin
  --[INTERACTS_WITH]--> Ibuprofen
  --[CAUSES_SIDE_EFFECT]--> Increased Bleeding Risk
  --[TREATS]--> Deep Vein Thrombosis
  --[BELONGS_TO_CLASS]--> Anticoagulant


> **Try in Neo4j Browser:**
>
> ```cypher
> -- All outgoing relationships from Warfarin
> MATCH (d:Drug {name: 'Warfarin'})-[r]->(m)
> RETURN type(r) AS rel, m.name AS target
> ```
>
> ```cypher
> -- Visual: full neighborhood of Warfarin
> MATCH path = (d:Drug {name: 'Warfarin'})-[r]-(m) RETURN path
> ```

## 5. Load Full Dataset & Build Graph

Now let's load the complete dataset (80 drugs, 244 interactions) and build the full knowledge graph using batch `UNWIND` + `MERGE`.

In [7]:
import json
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List

class DrugRecord(BaseModel):
    name: str
    aliases: List[str] = Field(default_factory=list)
    drug_class: str
    description: str
    treats: List[str] = Field(default_factory=list)
    side_effects: List[str] = Field(default_factory=list)
    contraindications: List[str] = Field(default_factory=list)

class InteractionRecord(BaseModel):
    drug_a: str
    drug_b: str
    severity: str
    mechanism: str
    effect: str
    recommendation: str

class TextPassage(BaseModel):
    source: str
    text: str

class DrugInteractionDataset(BaseModel):
    drugs: List[DrugRecord]
    interactions: List[InteractionRecord]
    text_passages: List[TextPassage] = Field(default_factory=list)

# Load
dataset_path = Path("../data/raw/drug_interactions.json")
with open(dataset_path) as f:
    raw = json.load(f)
dataset = DrugInteractionDataset(**raw)

print(f"Loaded: {len(dataset.drugs)} drugs, {len(dataset.interactions)} interactions, {len(dataset.text_passages)} text passages")
drug_classes = set(d.drug_class for d in dataset.drugs)
print(f"Drug classes ({len(drug_classes)}): {sorted(drug_classes)[:10]}...")

# Clean and rebuild
from loguru import logger

client.write("MATCH (n) DETACH DELETE n")

# Re-create constraints
for q in CONSTRAINT_QUERIES + INDEX_QUERIES:
    try:
        client.write(q)
    except:
        pass

# 1. Drug nodes
drug_batch = [{"name": d.name, "description": d.description, "aliases": d.aliases} for d in dataset.drugs]
client.batch_write("""
    UNWIND $batch AS row
    MERGE (d:Drug {name: row.name})
    SET d.description = row.description, d.aliases = row.aliases
""", drug_batch)
print(f"Created {len(drug_batch)} Drug nodes")

# 2. Condition nodes + TREATS
treats_batch = [{"drug": d.name, "condition": c} for d in dataset.drugs for c in d.treats]
if treats_batch:
    client.batch_write("""
        UNWIND $batch AS row
        MERGE (c:Condition {name: row.condition})
        MERGE (d:Drug {name: row.drug})
        MERGE (d)-[:TREATS]->(c)
    """, treats_batch)
    print(f"Created {len(set(t['condition'] for t in treats_batch))} Condition nodes, {len(treats_batch)} TREATS edges")

# 3. SideEffect nodes + CAUSES_SIDE_EFFECT
se_batch = [{"drug": d.name, "side_effect": se} for d in dataset.drugs for se in d.side_effects]
if se_batch:
    client.batch_write("""
        UNWIND $batch AS row
        MERGE (s:SideEffect {name: row.side_effect})
        MERGE (d:Drug {name: row.drug})
        MERGE (d)-[:CAUSES_SIDE_EFFECT]->(s)
    """, se_batch)
    print(f"Created {len(set(s['side_effect'] for s in se_batch))} SideEffect nodes, {len(se_batch)} CAUSES_SIDE_EFFECT edges")

# 4. Contraindications
contra_batch = [{"drug": d.name, "condition": ci} for d in dataset.drugs for ci in d.contraindications]
if contra_batch:
    client.batch_write("""
        UNWIND $batch AS row
        MERGE (c:Condition {name: row.condition})
        MERGE (d:Drug {name: row.drug})
        MERGE (d)-[:CONTRAINDICATED_FOR]->(c)
    """, contra_batch)
    print(f"Created {len(contra_batch)} CONTRAINDICATED_FOR edges")

# 5. DrugClass nodes + BELONGS_TO_CLASS
class_batch = [{"drug": d.name, "drug_class": d.drug_class} for d in dataset.drugs]
if class_batch:
    client.batch_write("""
        UNWIND $batch AS row
        MERGE (dc:DrugClass {name: row.drug_class})
        MERGE (d:Drug {name: row.drug})
        MERGE (d)-[:BELONGS_TO_CLASS]->(dc)
    """, class_batch)
    print(f"Created {len(set(c['drug_class'] for c in class_batch))} DrugClass nodes, {len(class_batch)} BELONGS_TO_CLASS edges")

# 6. Drug-Drug Interactions
ix_batch = [{
    "drug_a": ix.drug_a, "drug_b": ix.drug_b,
    "severity": ix.severity, "mechanism": ix.mechanism,
    "effect": ix.effect, "recommendation": ix.recommendation,
} for ix in dataset.interactions]
if ix_batch:
    client.batch_write("""
        UNWIND $batch AS row
        MERGE (a:Drug {name: row.drug_a})
        MERGE (b:Drug {name: row.drug_b})
        MERGE (a)-[r:INTERACTS_WITH]->(b)
        SET r.severity = row.severity, r.mechanism = row.mechanism,
            r.effect = row.effect, r.recommendation = row.recommendation, r.source = 'structured'
    """, ix_batch)
    print(f"Created {len(ix_batch)} INTERACTS_WITH edges")

print("\nFull graph built!")

Loaded: 80 drugs, 244 interactions, 19 text passages
Drug classes (23): ['ACE Inhibitor', 'ARB', 'Antiarrhythmic', 'Antibiotic', 'Anticoagulant', 'Anticonvulsant', 'Antidiabetic', 'Antiplatelet', 'Benzodiazepine', 'Beta-Blocker']...
Created 80 Drug nodes
Created 146 Condition nodes, 302 TREATS edges
Created 135 SideEffect nodes, 430 CAUSES_SIDE_EFFECT edges
Created 234 CONTRAINDICATED_FOR edges
Created 23 DrugClass nodes, 80 BELONGS_TO_CLASS edges
Created 244 INTERACTS_WITH edges

Full graph built!


> **Try in Neo4j Browser** — verify the full graph:
>
> ```cypher
> -- Node counts by label
> MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Relationship counts by type
> MATCH ()-[r]->() RETURN type(r) AS relationship, count(r) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Severity distribution
> MATCH ()-[r:INTERACTS_WITH]->() RETURN r.severity AS severity, count(r) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Visual: entire graph (may be large)
> MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 200
> ```
>
>```cypher
> -- Warfarin full neighborhood (same as pyvis above)
> MATCH path = (d:Drug {name: 'Warfarin'})-[r]-(m) RETURN path
> ```
>
> ```cypher
> -- Top 10 most-connected drugs
> MATCH (d:Drug)-[r]-() RETURN d.name AS drug, count(r) AS connections ORDER BY connections DESC LIMIT 10
> ```
>
> ```cypher
> -- All drugs in the NSAID class
> MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(dc:DrugClass {name: 'NSAID'}) RETURN d.name AS drug
> ```

## 7. LLM-Powered Entity Extraction

For unstructured text, we use GPT-4o with Pydantic structured output to extract entities and relationships automatically.

In [8]:
from pydantic import BaseModel, Field
from typing import List, Optional

# --- Extraction Schemas ---

class ExtractedEntity(BaseModel):
    """An entity extracted from unstructured text."""
    name: str = Field(..., description="Canonical name of the entity (e.g., 'Warfarin', 'Diabetes')")
    entity_type: str = Field(
        ..., description="One of: Drug, Condition, SideEffect, DrugClass"
    )
    aliases: List[str] = Field(
        default_factory=list,
        description="Alternative names or abbreviations (e.g., ['Coumadin'] for Warfarin)",
    )
    description: Optional[str] = Field(
        None, description="Brief description if mentioned in the text"
    )


class ExtractedRelationship(BaseModel):
    """A relationship extracted from unstructured text."""
    source: str = Field(..., description="Name of the source entity")
    source_type: str = Field(..., description="Type of the source entity")
    target: str = Field(..., description="Name of the target entity")
    target_type: str = Field(..., description="Type of the target entity")
    rel_type: str = Field(
        ...,
        description="One of: INTERACTS_WITH, TREATS, CAUSES_SIDE_EFFECT, CONTRAINDICATED_FOR, BELONGS_TO_CLASS",
    )
    evidence: str = Field(..., description="Quote from the text supporting this relationship")
    # NOTE: We use explicit typed fields instead of `properties: dict` because
    # OpenAI's structured output requires additionalProperties=false on all objects.
    severity: Optional[str] = Field(None, description="Severity for interactions: Major, Moderate, or Minor")
    mechanism: Optional[str] = Field(None, description="Pharmacological mechanism (for interactions)")


class ExtractionResult(BaseModel):
    """Complete extraction result from a text chunk."""
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)


# --- Load prompts from central config ---
from utils.prompts import EXTRACTION_SYSTEM_PROMPT, EXTRACTION_HUMAN_PROMPT

print("Extraction schemas and prompts defined.")

Extraction schemas and prompts defined.


In [9]:
from langchain_core.prompts import ChatPromptTemplate

def extract_from_text(text, llm=None):
    """Extract entities and relationships from a single text chunk."""
    if llm is None:
        llm = get_strong_llm()
    structured_llm = llm.with_structured_output(ExtractionResult)
    prompt = ChatPromptTemplate.from_messages([
        ("system", EXTRACTION_SYSTEM_PROMPT),
        ("human", EXTRACTION_HUMAN_PROMPT),
    ])
    chain = prompt | structured_llm
    return chain.invoke({"text": text})

# Demo: extract from first text passage
sample_text = dataset.text_passages[0].text
print(f"Input text:\n{sample_text[:300]}...\n")
result = extract_from_text(sample_text)
print(f"Extracted {len(result.entities)} entities, {len(result.relationships)} relationships")
for e in result.entities:
    print(f"  Entity: {e.name} ({e.entity_type})")
for r in result.relationships:
    print(f"  Rel: {r.source} --[{r.rel_type}]--> {r.target}")

Input text:
Warfarin (Coumadin) is known to interact with a remarkably wide range of medications due to its narrow therapeutic index and dependence on hepatic cytochrome P450 enzymes for metabolism. The S-enantiomer, which is 3-5 times more potent than the R-enantiomer, is primarily metabolized by CYP2C9. Drugs...

Extracted 16 entities, 13 relationships
  Entity: Warfarin (Drug)
  Entity: Amiodarone (Drug)
  Entity: Fluconazole (Drug)
  Entity: Metronidazole (Drug)
  Entity: Fluoxetine (Drug)
  Entity: Carbamazepine (Drug)
  Entity: Phenytoin (Drug)
  Entity: Rifampin (Drug)
  Entity: Hemorrhagic complications (SideEffect)
  Entity: Thromboembolic events (Condition)
  Entity: Antibiotics (DrugClass)
  Entity: Antifungals (DrugClass)
  Entity: Antiarrhythmics (DrugClass)
  Entity: Anticonvulsants (DrugClass)
  Entity: CYP2C9 (DrugClass)
  Entity: INR (Condition)
  Rel: Warfarin --[INTERACTS_WITH]--> Amiodarone
  Rel: Warfarin --[INTERACTS_WITH]--> Fluconazole
  Rel: Warfarin --[INTERAC

> **Try in Neo4j Browser** — if the extracted entities were ingested, you could query them:
>
> ```cypher
> -- Find all drugs that interact with any NSAID
> MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(dc:DrugClass {name: 'NSAID'})
> MATCH (d)-[r:INTERACTS_WITH]-(other:Drug)
> RETURN d.name AS nsaid, other.name AS interacts_with, r.severity
> ```
>
> ```cypher
> -- Drugs that cause bleeding-related side effects
> MATCH (d:Drug)-[:CAUSES_SIDE_EFFECT]->(se:SideEffect)
> WHERE toLower(se.name) CONTAINS 'bleed'
> RETURN d.name AS drug, se.name AS side_effect
> ```

## Summary

| Concept | What You Learned |
|---|---|
| Schema Design | 4 node labels, 5 relationship types |
| Cypher | MATCH, WHERE, RETURN, MERGE, parameterized queries |
| UNWIND + MERGE | Batch graph ingestion — fast and idempotent |
| LLM Extraction | GPT-4o + structured output for entity extraction |

**Next**: Notebook 02 — Agentic GraphRAG with LangGraph

In [10]:
client.close()
print("Done! Neo4j connection closed.")

Done! Neo4j connection closed.
